In [1]:
!pip -q install google-generativeai
!pip -q install sentence-transformers
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.1 MB/s eta 0:00:00


In [2]:
import faiss
import pickle
import numpy as np
import pandas as pd

from google.colab import files
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
index = faiss.read_index("/content/aeroassist_index.faiss")

with open("/content/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Vectors:", index.ntotal)
print("Metadata:", len(metadata))

Vectors: 9723
Metadata: 9723


In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
GOOGLE_API_KEY = "AQ.Ab8RN6KK4Zl4tvw6_JUQNVYB70J7Jr4ZctOQtpDQdHvFPPEmSg"

genai.configure(api_key=GOOGLE_API_KEY)

In [14]:
GOOGLE_API_KEY = "AQ.Ab8RN6KK4Zl4tvw6_JUQNVYB70J7Jr4ZctOQtpDQdHvFPPEmSg"

genai.configure(api_key=GOOGLE_API_KEY)

In [18]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [19]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, k)

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results

In [20]:
def build_prompt(query):

    docs = retrieve(query)

    context = "\n\n".join(
        [doc["text"] for doc in docs]
    )

    prompt = f"""
You are AeroAssist AI.

Answer ONLY using the information provided below.

If the answer is not present,
say:
"I couldn't find this information in the FAA manuals."

Context:

{context}

Question:

{query}

Answer:
"""

    return prompt

In [21]:
def ask_aeroassist(query):

    prompt = build_prompt(query)

    response = model.generate_content(prompt)

    return response.text

In [25]:
model = genai.GenerativeModel("gemini-3.6-flash")

In [26]:
response = model.generate_content("Say hello in one sentence.")
print(response.text)

Hello, I hope you are having a wonderful day!


In [27]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, k)

    retrieved_docs = []

    for idx in indices[0]:
        retrieved_docs.append(metadata[idx])

    return retrieved_docs

In [28]:
def build_prompt(query):

    docs = retrieve(query)

    context = ""

    for i, doc in enumerate(docs, start=1):

        context += f"""
Document {i}
Source: {doc['source']}

{doc['text']}

------------------------------------------
"""

    prompt = f"""
You are AeroAssist AI.

You are an expert aircraft maintenance assistant.

Answer ONLY using the FAA manual excerpts below.

If the answer is not available in the excerpts,
reply:

"I couldn't find enough information in the FAA manuals."

Do not invent facts.

FAA Manual Excerpts:

{context}

User Question:

{query}

Answer:
"""

    return prompt

In [29]:
def ask_aeroassist(query):

    prompt = build_prompt(query)

    response = model.generate_content(prompt)

    return response.text

In [30]:
question = "What is compressor stall?"

print(ask_aeroassist(question))

Based on the provided FAA manuals, a compressor stall is a condition in a turbine engine axial-flow compressor in which the angle of attack of one or more blades is excessive and the smooth airflow through the compressor is disrupted.


In [31]:
question = "What are the functions of a magneto?"

print(ask_aeroassist(question))

I couldn't find enough information in the FAA manuals.


In [32]:
question = "Explain aircraft corrosion inspection."

print(ask_aeroassist(question))

Based on the provided FAA manual excerpts, aircraft corrosion inspection involves the following steps, methods, and considerations:

### **Preparation and Access**
* **Cleaning:** All corrosion inspections begin with a thorough cleaning of the area to be inspected.
* **Gaining Access:** If inspection areas are obscured by structural members or equipment, access must be obtained by removing access panels and adjacent equipment, cleaning the area, and removing loose or cracked sealants and paints. 
* **Drain Holes:** Drain holes must be kept clean, as internal corrosion is most likely to occur in pockets and corners where moisture and salt spray accumulate.

---

### **Inspection Procedure**
1. **General Visual Inspection:** Uses a flashlight, inspection mirror, and a 5–10X magnifying glass to look for obvious defects and suspected areas.
2. **Detailed Inspection:** A detailed inspection follows for any damage or suspected areas found during the general inspection.

---

### **Inspection